### Imports

In [1]:
import os
from pathlib import Path
import json
import cv2
import supervision as sv
from tqdm import tqdm
import sys
import numpy as np


### Initialization

In [2]:
sys.path.insert(0, "../../")
from config import MEDIA_PATH, TEMP_PATH, CROPPED_PATH

sys.path.insert(0, "../../packages/python")
from models import cell_segmentation as segmentators

IMG_TARGET_SIDE = 200

JSON_PATH = os.path.join(TEMP_PATH, 'datasets_area_data.json')


2025-09-24 22:30:48.724156: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-24 22:30:48.747041: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-24 22:30:49.330253: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Functions

In [20]:
def crop_and_save_detections(images_dir: Path, annotations_path: Path, output_dir: Path, resize_factor, area_data) -> None:
    """
    Loads COCO annotations, crops the detected objects from images, and saves
    them into class-specific folders.

    Args:
        images_dir (Path): The path to the directory containing the images.
        annotations_path (Path): The path to the COCO JSON annotation file.
        output_dir (Path): The path to the directory where cropped images will be saved.
    """
    # Ensure the main output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the dataset using supervision
    print("Loading dataset...")
    dataset = sv.DetectionDataset.from_coco(
        images_directory_path=str(images_dir),
        annotations_path=str(annotations_path),
    )

    print(f"Found {len(dataset.classes)} classes: {dataset.classes}")

    # Iterate through the dataset with a progress bar
    for image_path, image, detections in tqdm(dataset):
        if image is None:
            continue
        image_name = Path(image_path).stem
        image_group = image_name[0]
        # image_side = area_data[image_group]['lado_cuadrado']
        # image_resize_factor = int(resize_factor * image_side)

        # Iterate through each detection in the image
        for i, detection in enumerate(detections):
            # The detection object contains xyxy, mask, confidence, class_id, etc.
            xyxy, _, _, class_id, _, _ = detection

            # Get the class name for the current detection
            class_name = dataset.classes[class_id].lower()

            # Create a directory for the class if it doesn't exist
            class_dir = output_dir / class_name 
            class_dir.mkdir(parents=True, exist_ok=True)

            # Crop the detection from the image using its bounding box
            x1, y1, x2, y2 = map(int, xyxy)
            cropped_image = image[y1:y2, x1:x2]

            # Ensure the cropped image is not empty before saving
            if cropped_image.size == 0:
                print(f"  - Skipping empty crop for detection {i} in {image_name}.png")
                continue

            cropped_image = cv2.resize(cropped_image, (IMG_TARGET_SIDE, IMG_TARGET_SIDE))
            # x, y, w, h = segmentators.CellMaskGenerator.adjust_bbox(segmentators.CellMaskGenerator, x1, y1, x2-x1, y2-y1, image_resize_factor*image_resize_factor, image.shape[1], image.shape[0])
            # cropped_image = cv2.resize(image[y:y+h, x:x+w], (IMG_TARGET_SIDE, IMG_TARGET_SIDE))


            # Generate a unique filename for the cropped image
            cropped_image_filename = f"{image_name}_{i}.png"
            cropped_image_path = class_dir / cropped_image_filename

            # Save the cropped image
            cv2.imwrite(str(cropped_image_path), cropped_image)

    print(f"\n✅ Processing complete. Cropped images are saved in '{output_dir}'.")


In [4]:
def find_images_with_black_markings(directory_path, black_threshold=10, white_threshold=245, percentage_threshold=1.0):
    """
    Identifies images in a directory that have a significant number of black pixels.

    Args:
        directory_path (str): The path to the directory containing images.
        black_threshold (int): Pixel intensity value (0-255). Pixels below this
                               value are considered 'black'. Defaults to 10.
        percentage_threshold (float): The percentage of black pixels required
                                      to classify an image as having markings.
                                      (e.g., 1.0 for 1%). Defaults to 1.0.

    Returns:
        list: A list of filenames for images that have significant black markings.
    """
    images_with_markings = []
    
    # A list of common image file extensions to check
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    print(f"Scanning directory: {directory_path}")

    # Iterate over every file in the directory
    for filename in os.listdir(directory_path):
        # Check for a valid image extension
        if not any(filename.lower().endswith(ext) for ext in valid_extensions):
            continue

        file_path = os.path.join(directory_path, filename)

        try:
            # Read the image using OpenCV
            image = cv2.imread(file_path)

            if image is None:
                print(f"Warning: Could not read image {filename}. Skipping.")
                continue

            # Convert the image to grayscale for easier processing
            gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            # Calculate the total number of pixels
            total_pixels = gray_image.size

            # Count the number of pixels that are 'black' (below the threshold)
            black_pixels = np.sum(gray_image <= black_threshold)

            white_pixels = np.sum(gray_image >= white_threshold)

            # Calculate the percentage of black pixels
            percentage_of_solid_pixels = ((black_pixels + white_pixels) / total_pixels) * 100

            # If the percentage exceeds the threshold, add it to our list
            if percentage_of_solid_pixels > percentage_threshold:
                # print(f"Found markings in {filename} ({percentage_of_solid_pixels:.2f}% black pixels)")
                images_with_markings.append(filename)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    return images_with_markings



### Crops images from coco annotations

In [22]:
# IMAGES_DIR = Path(os.path.join(MEDIA_PATH, 'nuevos datasets','Mitosis.v1i.coco', 'train'))
# ANNOTATIONS_PATH = Path(os.path.join(MEDIA_PATH, 'nuevos datasets','Mitosis.v1i.coco', 'train', '_annotations.coco.json'))
# OUTPUT_DIR = Path(os.path.join(MEDIA_PATH, 'nuevos datasets','Mitosis.v1i.coco'))

IMAGES_DIR = Path(os.path.join(MEDIA_PATH, "images", "ina", "tagged_images", 'input'))
ANNOTATIONS_PATH = Path(os.path.join(MEDIA_PATH, 'images', 'ina', 'tagged_images', 'corte-27-02-2024.json'))
OUTPUT_DIR = Path(os.path.join(MEDIA_PATH, "images", "ina", "tagged_images"))

with open(JSON_PATH, 'r') as f: #json with the information of the filename of the images
    area_data = json.load(f)

resize_factor = IMG_TARGET_SIDE/area_data['INA']['lado_cuadrado']

crop_and_save_detections(
    images_dir=IMAGES_DIR,
    annotations_path=ANNOTATIONS_PATH,
    output_dir=OUTPUT_DIR,
    resize_factor=resize_factor,
    area_data = area_data
)


Loading dataset...


[ WARN:0@1349.019] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00001.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1349.019] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00002.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1349.019] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00003.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1349.019] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_00004.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1349.019] global loadsave.cpp:241 findDecoder imread_('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/001_0

KeyboardInterrupt: 

### Remove crops with black or white spaces

In [15]:
IMAGE_DIRECTORY = os.path.join(CROPPED_PATH, 'd')
IMAGE_DIRECTORY = os.path.join(MEDIA_PATH, 'images','onion_cell_merged', 'images', 'crops', 'd')

# You can adjust these thresholds as needed
BLACK_PIXEL_INTENSITY_THRESHOLD = 10  # How dark a pixel must be to be 'black' (0-255)
WHITE_PIXEL_INTENSITY_THRESHOLD = 245 # How brigth a pixel must be to be 'white' (0-255)
BLACK_PIXEL_PERCENTAGE = 15.0    # What percentage of the image needs to be black or white

marked_images = find_images_with_black_markings(
    IMAGE_DIRECTORY,
    black_threshold=BLACK_PIXEL_INTENSITY_THRESHOLD,
    white_threshold=WHITE_PIXEL_INTENSITY_THRESHOLD,
    percentage_threshold=BLACK_PIXEL_PERCENTAGE
)

if marked_images:
    print("The following images appear to have black markings:")
    print(f"- {sorted(marked_images)}")
    print(f"- {len(sorted(marked_images))}")
    for img_name in sorted(marked_images):
        os.remove(os.path.join(IMAGE_DIRECTORY, img_name))
else:
    print("No images with significant black markings were found.")

Scanning directory: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/onion_cell_merged/images/crops/d
The following images appear to have black markings:
- ['A_102_69.png', 'A_102_70.png', 'A_107_122.png', 'A_10_25.png', 'A_112_11.png', 'A_115_82.png', 'A_117_114.png', 'A_117_115.png', 'A_120_97.png', 'A_121_116.png', 'A_123_87.png', 'A_126_19.png', 'A_17_3.png', 'A_17_42.png', 'A_17_47.png', 'A_17_5.png', 'A_18_3.png', 'A_18_56.png', 'A_22_18.png', 'A_230_4.png', 'A_235_20.png', 'A_248_25.png', 'A_248_3.png', 'A_248_4.png', 'A_249_1.png', 'A_24_31.png', 'A_251_5.png', 'A_252_5.png', 'A_252_7.png', 'A_255_4.png', 'A_256_5.png', 'A_268_16.png', 'A_26_8.png', 'A_274_4.png', 'A_27_3.png', 'A_27_50.png', 'A_280_2.png', 'A_28_53.png', 'A_298_6.png', 'A_314_3.png', 'A_331_0.png', 'A_331_19.png', 'A_33_27.png', 'A_33_33.png', 'A_33_58.png', 'A_350_2.png', 'A_36_33.png', 'A_41_39.png', 'A_49_4.png', 'A_49_41.png', 'A_51_3.png', 'A_53_3.png', 'A_59_21.png', 'A_59_37.png', 'A_61_57.png

### Unused code

In [ ]:
from tqdm import tqdm

"""
Codigo para renombrar las imagenes taggeadas para que tengan el mismo nombre que en el json
"""

IMAGES_DIR = (os.path.join(MEDIA_PATH, "images", "ina", "tagged_images", 'input'))
ANNOTATIONS_PATH = (os.path.join(MEDIA_PATH, 'images', 'ina', 'tagged_images', 'corte-27-02-2024.json'))
OUTPUT_DIR = (os.path.join(MEDIA_PATH, "images", "ina", "tagged_images"))

images_paths = [IMAGES_DIR + '/' + item for item in sorted(os.listdir(IMAGES_DIR))]

with open(ANNOTATIONS_PATH, 'r') as f: #json with the information of the filename of the images
    data = json.load(f)

for image in tqdm(images_paths):
    image_id = os.path.basename(image).split('.')[0]

    if image_id.isnumeric() == False:
        print('Ignoring non-numeric image:', image)
        continue

    value = data['images'][int(image_id) - 1]['file_name']
    os.rename(image, f'{IMAGES_DIR}/{value}')

100%|██████████| 58/58 [00:00<00:00, 68934.44it/s]

Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/004_00094.jpg
Ignoring non-numeric image: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/images/ina/tagged_images/input/004_00095.jpg


In [ ]:
import cv2
import os
from tqdm import tqdm

def resize_images_in_directory(dataset_path, output_dir, target_size=(200, 200)):
    """
    Searches for images starting with 'IMG' in a directory, resizes them,
    and saves them to an output directory.

    Args:
        dataset_path (str): The path to the directory containing the images.
        output_dir (str): The path to the directory where resized images will be saved.
        target_size (tuple): A tuple (width, height) for the resized images.
    """
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # Check if the dataset path exists
    if not os.path.isdir(dataset_path):
        print(f"Error: The directory '{dataset_path}' was not found.")
        return

    print(f"Searching for images in '{dataset_path}'...")

    # Loop through all the files in the source directory
    for filename in tqdm(os.listdir(dataset_path)):
        # Check if the file name starts with 'IMG' and is a supported image format
        if filename.startswith('IMG') and filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_filepath = os.path.join(dataset_path, filename)

            # Read the image using OpenCV
            image = cv2.imread(input_filepath)

            # Check if the image was loaded successfully
            if image is None:
                print(f"Warning: Could not read image {input_filepath}. Skipping.")
                continue

            # Resize the image
            # cv2.INTER_AREA is generally good for shrinking images.
            resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)

            # Construct the output file path
            output_filepath = os.path.join(output_dir, filename)

            # Save the resized image
            cv2.imwrite(output_filepath, resized_image)
            print(f"Resized and saved '{filename}' to '{output_filepath}'")

    print("\nProcessing complete.")

DATASET_PATH = os.path.join(CROPPED_PATH, 'd') # Run again for 'not' dataset
resize_images_in_directory(DATASET_PATH, DATASET_PATH)

